# EM sections and synapse counts of the MICrONS volume (MICrONS Consortium 2025)

**Paper.** The MICrONS Consortium (2025). Functional connectomics spanning multiple areas of mouse
visual cortex. *Nature* 640, 435-447.
[doi:10.1038/s41586-025-08790-w](https://doi.org/10.1038/s41586-025-08790-w). The claims file,
`claims/000402/10.1038-s41586-025-08790-w.yaml`, was written from the open access version of record
(PMC11981939).

**Data.** The two claims here are `data: elsewhere`: the EM imagery and the synapse tables of the
MICrONS volume, which the BossDB open data bucket serves without an account
(`s3://bossdb-open-data/iarpa_microns/minnie/`). The imagery of each subvolume is in neuroglancer
precomputed format (`minnie35/em`, `minnie65/em`); the helper reads its sharded chunks directly.
The synapse tables are gzipped CSV files, streamed once and counted without being kept. Dandiset
000402 (version 0.230307.2132), the entry's dataset, holds no EM data; the helper names it because
this notebook belongs to the entry on it. The other analyses of the entry are in
`reproduce_microns_2025.ipynb`.

This notebook was added in the second revision, after the verifier's second round. Both claims were
verified as blocked (`too_large`); the outcomes found here are recorded as proposed changes, for a
third verification round.

**Random numbers.** None are used.

| Claim | Data | Tested here |
|---|---|---|
| em-sections | elsewhere | yes, from the imagery's metadata and its sections with data |
| synapse-detection | elsewhere | the two counts; precision, recall and F1 are blocked (no manual annotations found) |

## Data check

In [ ]:
import time
import zlib

import numpy as np
import pandas as pd

import microns

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
# The files on the BossDB bucket that this notebook and the attempt record cite, with their sizes
listing = pd.concat([microns.boss_list(p) for p in (
    "minnie65/synapse_graph/", "minnie35/synapse_graph/", "functional_data/two_photon_structural_stacks/",
    "functional_coregistration/", "proofreading_status/", "minnie65/em/info", "minnie35/em/info")])
listing = listing[~listing.key.str.endswith("/")]
listing["MB"] = (listing.bytes / 1e6).round(2)
print(listing[["key", "MB"]].to_string(index=False))
# EM imagery: every scale of each subvolume
for vol in ("minnie35", "minnie65"):
    info = microns._session.get(f"{microns.BOSS_ROOT}{vol}/em/info", timeout=60).json()
    print(f"\n{vol}/em: {info['data_type']}, {info['num_channels']} channel; scales (nm per voxel, offset, size, chunk, encoding):")
    for s in info["scales"]:
        print(f"  {s['key']:>15}  offset {s['voxel_offset']}  size {s['size']}  chunk {s['chunk_sizes'][0]}  {s['encoding']}")

Each subvolume's imagery has 9 scales. The z axis counts 40 nm sections at the finest four scales
(8 to 64 nm per pixel in x and y) and merges sections at the coarser ones, so 64 nm per pixel
(`64_64_40`) is the coarsest scale that keeps every section. The files cited in the attempt record
are listed with their sizes: the synapse table exports of subvolume 65 (versions 1078, 1181 and 1300,
about 20 GB each), the unversioned table `synapses_pni_2.csv` (51 GB), the connection table of
version 1300, the synapse table of subvolume 35 (`assigned.csv.gz`, 13.8 GB), the structural stacks,
the 2021 match release and the 2021 proofreading release.

## em-sections

> The block was cut into 27,972 sections of 40 nm, 26,652 were imaged by five autoTEMs, and an 800
> um region of sections 7,931 to 27,904 was processed, split into two subvolumes of about 35% and 65%
> of the sections.

Criterion: the released imagery and its metadata give the sections of each subvolume (7,931 to
14,815 and 14,816 to 27,904). The processed region is 27,904 - 7,931 + 1 sections (expected 19,974,
tolerance 0), and the share of subvolume 35 is 35 +- 3.5%. Counts of sections cut and imaged are
reported if the released metadata give them.

The metadata give each subvolume's bounds as whole chunks of 64 sections at the 64 nm scale
(`voxel_offset` and `size` above). The first and last section that hold data are read from the
imagery: at the 64 nm scale, the chunk layers at each end of each subvolume are sampled on a grid of
every 8th chunk in x and y (52 x 44 = 2,288 chunks a layer), and a section has data when any sampled
chunk has a non-zero voxel in it. The grid is a choice of this attempt; at the last section of
subvolume 65 it is checked against a sparser grid and against the 16 nm scale.

In [ ]:
started = time.time()
sc35, sc65 = microns.em_scale("minnie35"), microns.em_scale("minnie65")
b = {}
for vol, s in (("minnie35", sc35), ("minnie65", sc65)):
    z0, nz = s["voxel_offset"][2], s["size"][2]
    b[vol] = (z0, z0 + nz - 1)
    print(f"{vol} metadata: sections {z0} to {z0 + nz - 1} ({nz})")
lo, hi = b["minnie35"][0], b["minnie65"][1]
print(f"Metadata, union: {lo} to {hi}, {hi - lo + 1} sections; overlap of the two subvolumes: "
      f"{b['minnie35'][1] - b['minnie65'][0] + 1} sections ({b['minnie65'][0]} to {b['minnie35'][1]})")
print(f"Metadata, share of subvolume 35 with the split at the start of subvolume 65: "
      f"{100 * (b['minnie65'][0] - lo) / (hi - lo + 1):.2f}%")

# Sections with data at both ends of each subvolume
layers = {"minnie35": (0, (14752 - sc35["voxel_offset"][2]) // 64, (14816 - sc35["voxel_offset"][2]) // 64),
          "minnie65": (0, (27808 - sc65["voxel_offset"][2]) // 64, (27872 - sc65["voxel_offset"][2]) // 64)}
has = {}
for vol, gzs in layers.items():
    for gz in gzs:
        z0, n, counts = microns.em_sections_with_data(vol, gz, step=8)
        for i, c in enumerate(counts):
            has[(vol, z0 + i)] = int(c)
        print(f"{vol}, sections {z0}-{z0 + len(counts) - 1}, {n} chunks sampled; sampled chunks with data per section:")
        print("  ", counts.tolist())
first = {v: min(z for (vv, z), c in has.items() if vv == v and c > 0) for v in layers}
last = {v: max(z for (vv, z), c in has.items() if vv == v and c > 0) for v in layers}
for v in layers:
    print(f"{v}: first section with data {first[v]}, last {last[v]} (the last sampled section is {max(z for (vv, z) in has if vv == v)})")
print(f"Elapsed: {time.time() - started:.0f} s")

In [ ]:
# Checks of the last section of subvolume 65: a sparser grid at 64 nm, and 16 chunks at 16 nm
z0, n, counts = microns.em_sections_with_data("minnie65", layers["minnie65"][2], step=16)
print(f"64 nm, every 16th chunk ({n} chunks): last section with data {z0 + int(np.where(counts > 0)[0].max())}")
s16 = microns.em_scale("minnie65", "16_16_40")
grid = [-(-s // c) for s, c in zip(s16["size"], s16["chunk_sizes"][0])]
gz = (27872 - s16["voxel_offset"][2]) // 64
lasts = []
for x in range(grid[0] // 4, 3 * grid[0] // 4, grid[0] // 8):
    for y in range(grid[1] // 4, 3 * grid[1] // 4, grid[1] // 8):
        a = microns.em_chunk("minnie65", s16, (x, y, gz))
        if a is not None and (a > 0).any():
            lasts.append(s16["voxel_offset"][2] + gz * 64 + int(np.where((a.reshape(a.shape[0], -1) > 0).any(axis=1))[0].max()))
print(f"16 nm, 16 chunks in the centre of the volume: {len(lasts)} with data; last section with data in each: {sorted(set(lasts))}")

In [ ]:
start, end = first["minnie35"], last["minnie65"]
n_data = end - start + 1
print(f"Sections with data: subvolume 35 {first['minnie35']} to {last['minnie35']}, subvolume 65 {first['minnie65']} to {last['minnie65']}")
print(f"Processed region from the imagery: {start} to {end}, {n_data} sections (expected 19,974, from 7,931 to 27,904)")
print(f"Processed region from the metadata bounds: {lo} to {hi}, {hi - lo + 1} sections")
rows = [("imagery, subvolume 35 up to the first section of subvolume 65", first["minnie65"] - start, n_data),
        ("imagery, all sections of subvolume 35 with data", last["minnie35"] - first["minnie35"] + 1, n_data),
        ("metadata bounds, split at the start of subvolume 65", b["minnie65"][0] - lo, hi - lo + 1),
        ("claims file split (7,931 to 14,815 of 7,931 to 27,904)", 14815 - 7931 + 1, 27904 - 7931 + 1)]
for name, k, n in rows:
    print(f"  share of subvolume 35, {name}: {k} of {n} = {100 * k / n:.2f}% (expected 31.5 to 38.5)")
print("Sections cut (27,972) and imaged (26,652): not in the metadata above.")

**Result: partially reproduced (proposed; the recorded outcome stays blocked until a third round of
verification).** The imagery of subvolume 35 starts at section 7,931, as the paper says. The
imagery of subvolume 65 ends at section 27,882, not 27,904, in every sampled chunk with data, at 64
nm and at 16 nm, so the region with data spans 7,931 to 27,882, 19,952 sections against the
expected 19,974 (tolerance 0). The metadata bounds, whole chunks of 64 sections, give 7,904 to
27,903, 20,000 sections, which also differs. The share of subvolume 35 is 34.55% to 34.67% by the
imagery and 34.56% by the metadata bounds, within 35 +- 3.5%. The two subvolumes overlap: the
imagery of subvolume 35 runs to 14,847 and that of subvolume 65 starts at 14,825, while the claims
file splits them at 14,815 and 14,816. The metadata do not give the sections cut or imaged.

## synapse-detection

> Automated detection found 524 million synaptic clefts (186 million in subvolume 35, 337 million in
> subvolume 65), with 96% precision and 89% recall against 8,611 manually annotated synapses in 70
> subvolumes, and 98% partner assignment accuracy on 191 held-out synapses.

Criterion: counts from the released synapse tables (expected 337 +- 33.7 million in subvolume 65 and
186 +- 18.6 million in subvolume 35). Precision and recall can be tested only if the manual
annotations of the 70 test subvolumes are public.

The paper names no materialization version. Subvolume 65 is counted on the export of version 1181,
the version the paper's code reads (claims file header, "its code uses version 1181"); the file has no header line, so each line is a synapse. Subvolume 35 has one
table on the bucket, `minnie35/synapse_graph/assigned.csv.gz` (2021), whose first line is a header.
Each file is streamed once; a broken stream resumes at the byte reached. Counting both took about 11
minutes while preparing.

In [ ]:
started = time.time()
syn = {}
for name, path, header in (("subvolume 65, version 1181", "minnie65/synapse_graph/cave_exports/v1181/synapses_pni_2_v1_filtered_view.csv.gz", 0),
                           ("subvolume 35", "minnie35/synapse_graph/assigned.csv.gz", 1)):
    print(name, path)
    head = microns._range(microns.BOSS_ROOT + path, 0, 4096)
    print("  first line:", zlib.decompressobj(16 + zlib.MAX_WBITS).decompress(head).split(b"\n")[0][:200].decode())
    lines, got = microns.count_csv_gz_lines(microns.BOSS_ROOT + path, report_gb=5)
    syn[name] = lines - header
    print(f"  {got / 1e9:.2f} GB read, {lines} lines, {syn[name]} synapses")
s65, s35 = syn["subvolume 65, version 1181"], syn["subvolume 35"]
print(f"\nSubvolume 65: {s65 / 1e6:.1f} million (expected 337 +- 33.7): {abs(s65 / 1e6 - 337) <= 33.7}")
print(f"Subvolume 35: {s35 / 1e6:.1f} million (expected 186 +- 18.6): {abs(s35 / 1e6 - 186) <= 18.6}")
print(f"Both: {(s65 + s35) / 1e6:.1f} million (the paper's total, 524 million)")
print("Manual annotations of the 70 test subvolumes: none among the files listed above.")
print(f"Elapsed: {time.time() - started:.0f} s")

**Result: partially reproduced (proposed; the recorded outcome stays blocked until a third round of
verification).** The version 1181 export of subvolume 65 holds 337.3 million synapses and the table
of subvolume 35 holds 186.3 million, both within tolerance of the paper's 337 and 186 million, and
together 523.6 million against 524 million. Precision, recall and F1 are not tested: no manual
synapse annotations were found on the bucket.

## Summary

| Claim | Data | Outcome |
|---|---|---|
| em-sections | elsewhere | blocked: too large, as verified; proposed partially reproduced (share of subvolume 35 within tolerance; 19,952 sections with data against 19,974) |
| synapse-detection | elsewhere | blocked: too large, as verified; proposed partially reproduced (both counts within tolerance; precision and recall not tested) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** The first and last sections are read from the
imagery at 64 nm per pixel on a grid of sampled chunks, not from a list of sections, which the
release does not give. The paper names no version of the synapse table; subvolume 65 is counted at
version 1181 and subvolume 35 on its only public table.

**What was found about the data.** The imagery of subvolume 65 ends at section 27,882, 22 sections
before the paper's 27,904, and the imagery of the two subvolumes overlaps from 14,825 to 14,847. The
metadata bounds are whole chunks and do not give the processed sections.